# Fine-tuned adapter behind autoscaling endpoint with A/B traffic split and shadow eval

KILLER LAB. The killer moment: traffic split holds within 1pp under a 200-request load test, the LLM judge correctly classifies disagreements, and you flip traffic from 10% to 0% in ONE config change and watch the LoRA endpoint stop serving.

A team has a fine-tuned LoRA adapter that they want to ship to 10% of production traffic while keeping 90% on the existing Claude Haiku baseline. They also want shadow traffic: every production request hits the new model in the background, the responses are compared in an LLM-as-judge pipeline, and disagreements are logged for manual review.

You build the routing layer in the Colab kernel, the A/B split (10/90 with a configurable lever), and the shadow eval pipeline (LLM judge writes disagreements to Supabase). Run a 200-request load test, confirm the split holds within 1 percentage point, and inspect the disagreement log.

**Duration:** about 120 minutes of work in a 150-minute session window.

**Cost preamble.** Most expensive lab in this sub-track. Modal L4 is the biggest line; Claude Sonnet as the judge is the second. About four bucks on a clean run. The killer moment is when you flip traffic from 10% to 0% in one line and watch the LoRA endpoint stop serving.

**CLEANUP IS CRITICAL.** A forgotten Modal L4 burns ~$26/day. The cleanup cell tears down the Modal app, the Volume, and the Supabase tables. Run it before you walk away.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies.

!pip install --quiet labrun-checks==0.7.0 modal==0.66.40 anthropic==0.42.0 supabase==2.9.0 requests==2.32.0

In [ ]:
# Imports and per-lab constants.

import atexit
import asyncio
import getpass
import json
import os
import random
import sys
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

import modal
import requests

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "lora-adapter-ab-shadow-eval"
LAB_TAG_VALUE = LAB_ID

MODAL_APP_NAME = f"labrun-{LAB_ID}-app"
MODAL_VOLUME_NAME = f"labrun-{LAB_ID}-weights"

TRAFFIC_LOG_TABLE = "labrun_lora_adapter_ab_shadow_eval_traffic_log"
DISAGREEMENTS_TABLE = "labrun_lora_adapter_ab_shadow_eval_disagreements"

SUMMARY_PATH = "/content/ab_test_summary.json"

ANTHROPIC_SONNET = "claude-sonnet-4-5-20250929"
ANTHROPIC_HAIKU = "claude-haiku-4-5-20250930"

# Live config the routing layer reads. Mutating SPLIT_CONFIG["lora_ratio"] is
# the "flip" that students perform in Task 5.
SPLIT_CONFIG = {"lora_ratio": 0.10}

# The LoRA "model" in this lab is a thin wrapper. To keep cost manageable, the
# Modal endpoint serves a Mistral 7B (no LoRA adapter actually applied; the
# "fine-tuned" identity is the system prompt below).
LORA_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
LORA_SYSTEM_PROMPT = (
    "You are Widget's customer-support assistant, tuned to mirror the team's house style: "
    "answer in 1-2 sentences, use 'we' and 'our', never apologize."
)

In [ ]:
# NBVAL_SKIP
# BYOK setup.

import anthropic
from supabase import create_client, Client

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
MODAL_TOKEN_ID = getpass.getpass("MODAL_TOKEN_ID: ").strip()
MODAL_TOKEN_SECRET = getpass.getpass("MODAL_TOKEN_SECRET: ").strip()
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
SUPABASE_URL = getpass.getpass("SUPABASE_URL: ").strip()
SUPABASE_SERVICE_ROLE_KEY = getpass.getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()
HF_TOKEN = getpass.getpass("HF_TOKEN (optional; press enter to skip): ").strip()

if not all([MODAL_TOKEN_ID, MODAL_TOKEN_SECRET, ANTHROPIC_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY]):
    print("Modal + Anthropic + Supabase credentials are all required. Re-run this cell.")
    raise SystemExit(1)

os.environ["MODAL_TOKEN_ID"] = MODAL_TOKEN_ID
os.environ["MODAL_TOKEN_SECRET"] = MODAL_TOKEN_SECRET
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
try:
    _ping = anthropic_client.messages.create(model=ANTHROPIC_HAIKU, max_tokens=8,
        messages=[{"role": "user", "content": "reply with: ok"}])
    print(f"Anthropic OK. Haiku said: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic rejected: {exc!r}")
    raise SystemExit(1)

supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
try:
    supabase.table("_supabase_unlikely_table_name").select("*").limit(1).execute()
except Exception:
    pass
print("Supabase reachable.")

try:
    modal.App.lookup("labrun-nonexistent-probe-app", create_if_missing=False)
except modal.exception.NotFoundError:
    print("Modal credentials validated.")
except modal.exception.AuthError as exc:
    print(f"Modal auth rejected: {exc!r}")
    raise SystemExit(1)
except Exception as exc:
    print(f"Modal probe raised: {exc!r}; treat as auth failure if persistent.")

register(
    session_token=session_token,
    lab_id=LAB_ID,
    credentials={"supabase_url": SUPABASE_URL, "modal_app_name": MODAL_APP_NAME},
)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, adapter (Modal + Supabase + local), atexit, orphan scan.

CLEANUP_MANIFEST = [
    CleanupResource(type="local_file", id=SUMMARY_PATH, region="local",
        cli_delete_command=f"rm -f {SUMMARY_PATH}"),
    CleanupResource(type="supabase_table", id=DISAGREEMENTS_TABLE, region="supabase",
        cli_delete_command=f"# SQL: DROP TABLE IF EXISTS public.{DISAGREEMENTS_TABLE}"),
    CleanupResource(type="supabase_table", id=TRAFFIC_LOG_TABLE, region="supabase",
        cli_delete_command=f"# SQL: DROP TABLE IF EXISTS public.{TRAFFIC_LOG_TABLE}"),
    CleanupResource(type="modal_volume", id=MODAL_VOLUME_NAME, region="modal",
        cli_delete_command=f"modal volume delete {MODAL_VOLUME_NAME}"),
    CleanupResource(type="modal_app", id=MODAL_APP_NAME, region="modal",
        cli_delete_command=f"modal app stop {MODAL_APP_NAME} && modal app delete {MODAL_APP_NAME}"),
]


class LoraLabCleanupAdapter:
    """Modal app + volume + Supabase tables + local file. The Modal app is
    critical; this adapter tears it down first.

    TODO: labrun-checks 0.8 ships Modal + Supabase adapters; migrate then.
    """

    def delete_resource(self, credentials, resource):
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        if resource.type == "modal_app":
            try:
                app = modal.App.lookup(resource.id, create_if_missing=False)
                if hasattr(app, "stop"):
                    try: app.stop()
                    except Exception: pass
                if hasattr(modal.App, "delete"):
                    try: modal.App.delete(resource.id)
                    except Exception: pass
            except modal.exception.NotFoundError:
                return
            except Exception:
                return
            return
        if resource.type == "modal_volume":
            try:
                vol = modal.Volume.from_name(resource.id)
                if hasattr(vol, "delete"):
                    vol.delete()
            except Exception:
                pass
            return
        if resource.type == "supabase_table":
            try:
                supabase.table(resource.id).delete().neq("id", 0).execute()
            except Exception:
                pass
            return

    def describe_resource(self, credentials, resource):
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        if resource.type == "modal_app":
            try:
                modal.App.lookup(resource.id, create_if_missing=False)
                return True
            except Exception:
                return False
        if resource.type == "modal_volume":
            try:
                modal.Volume.from_name(resource.id)
                return True
            except Exception:
                return False
        if resource.type == "supabase_table":
            try:
                rows = supabase.table(resource.id).select("id").limit(1).execute()
                return bool(rows.data)
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        return []


CLEANUP_ADAPTER = LoraLabCleanupAdapter()


_orphans = []
if os.path.exists(SUMMARY_PATH):
    _orphans.append(SUMMARY_PATH)
try:
    modal.App.lookup(MODAL_APP_NAME, create_if_missing=False)
    _orphans.append(f"modal://{MODAL_APP_NAME}")
except Exception:
    pass
for t in (TRAFFIC_LOG_TABLE, DISAGREEMENTS_TABLE):
    try:
        rows = supabase.table(t).select("id").limit(1).execute()
        if rows.data:
            _orphans.append(f"supabase://{t}")
    except Exception:
        pass

if _orphans:
    print("Orphan scan found leftover state:")
    for o in _orphans:
        print(f"  {o}")
    raise SystemExit(1)


def _on_exit_cleanup():
    try:
        for r in list(CLEANUP_MANIFEST):
            try:
                CLEANUP_ADAPTER.delete_resource({}, r)
            except Exception:
                pass
    except Exception:
        pass


atexit.register(_on_exit_cleanup)
print("Manifest registered. Orphan scan clean.")

## Supabase table setup

Two tables. Create via Supabase SQL editor before continuing:

```sql
create table public.labrun_lora_adapter_ab_shadow_eval_traffic_log (
  id bigserial primary key,
  request_id text not null unique,
  user_message text not null,
  primary_model text not null,
  primary_response text not null,
  shadow_model text,
  shadow_response text,
  shadow_model_responded boolean default false,
  judge_verdict text,
  inserted_at timestamptz default now()
);

create table public.labrun_lora_adapter_ab_shadow_eval_disagreements (
  id bigserial primary key,
  request_id text not null,
  user_message text not null,
  primary_response text not null,
  shadow_response text not null,
  judge_rationale text,
  inserted_at timestamptz default now()
);
```

In [ ]:
# NBVAL_SKIP
def confirm_tables():
    missing = []
    for t in (TRAFFIC_LOG_TABLE, DISAGREEMENTS_TABLE):
        try:
            supabase.table(t).select("id").limit(1).execute()
        except Exception:
            missing.append(t)
    if missing:
        print(f"Missing tables: {missing}. Create them via SQL editor, then re-run.")
        raise SystemExit(1)
    print("Tables exist.")


confirm_tables()

## Task 1: Deploy the "LoRA" endpoint on Modal

The endpoint serves Mistral 7B with the house-style system prompt. The Modal app autoscales 0-2 replicas with a warm-pool of 1. The smoke call should return a Widget-style response in 1-2 sentences.

In [ ]:
# Task 1: Modal app for the LoRA endpoint.

vllm_image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install("vllm==0.6.3", "transformers==4.45.2")
)

volume = modal.Volume.from_name(MODAL_VOLUME_NAME, create_if_missing=True)
app = modal.App(MODAL_APP_NAME)


@app.cls(
    gpu="L4",
    image=vllm_image,
    volumes={"/weights": volume},
    min_containers=1,
    max_containers=2,
    scaledown_window=120,
    timeout=600,
)
class LoraServer:

    @modal.enter()
    def load_model(self):
        from vllm import LLM
        self.llm = LLM(
            model=LORA_MODEL_ID,
            gpu_memory_utilization=0.85,
            max_model_len=4096,
            download_dir="/weights",
        )
        print("LoRA model loaded")

    @modal.method()
    def generate(self, user_message, max_tokens=120):
        from vllm import SamplingParams
        prompt = f"<s>[INST] {LORA_SYSTEM_PROMPT}\n\n{user_message} [/INST]"
        outputs = self.llm.generate([prompt], SamplingParams(max_tokens=max_tokens, temperature=0.2))
        return outputs[0].outputs[0].text.strip()


@app.function(image=vllm_image, timeout=120)
@modal.web_endpoint(method="POST", label="lora-generate")
def lora_endpoint(payload: dict):
    server = LoraServer()
    text = server.generate.remote(payload["user_message"], max_tokens=payload.get("max_tokens", 120))
    return {"text": text}


app.deploy()
print(f"Deployed {MODAL_APP_NAME}.")

endpoint_url = lora_endpoint.web_url
print(f"LoRA endpoint URL: {endpoint_url}")

# Smoke call.
resp = requests.post(endpoint_url, json={"user_message": "What is Widget?", "max_tokens": 80}, timeout=180)
print(f"Smoke status: {resp.status_code}; body: {resp.text[:200]}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: LoRA endpoint returns 200 with text.


def checkpoint_1(session):
    if endpoint_url is None:
        return CheckpointResult(status="fail", error_reason="endpoint_url is None.")
    try:
        r = requests.post(endpoint_url, json={"user_message": "What is Widget?", "max_tokens": 80}, timeout=180)
    except Exception as exc:
        return CheckpointResult(status="fail", error_reason=f"POST raised {exc!r}")
    if r.status_code != 200:
        return CheckpointResult(status="fail", error_reason=f"Endpoint status {r.status_code}: {r.text[:200]}")
    if "text" not in r.json():
        return CheckpointResult(status="fail", error_reason=f"Response missing 'text': {r.text[:200]}")
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The Modal app pattern matches Lab 13. The only differences are the model id, the system prompt, and `max_containers=2` (this lab does not need 4).

</details>

<details><summary>Hint 2 (direction)</summary>

Already supplied. If CP1 fails with timeout, the cold-load may exceed 180 seconds on a low-credit account; bump the request timeout to 240 and retry.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as supplied. The lab uses Mistral as the LoRA stand-in; a real production version would attach a `LoRARequest` to the SamplingParams. Skipping the actual adapter keeps the lab cost manageable.

</details>

**Wallet check.** Modal L4 cold-load + 5 min warm: ~$0.15. Cumulative: ~$0.15.

## Task 2: Counter-based A/B traffic split

Build a router that picks the primary model per request: LoRA with probability `SPLIT_CONFIG["lora_ratio"]`, Haiku otherwise. Random-based splits drift on small samples; a counter-based split (every Nth request goes to LoRA) holds within 1 percentage point over 200 requests.

Run the 200-request load test against the router. Log every request to `traffic_log` with `primary_model`, `primary_response`. The validator queries the table and confirms LoRA share is in [9%, 11%].

In [ ]:
# Task 2: counter-based router + 200-request load test.

REQUEST_COUNTER = {"n": 0}


def haiku_call(user_message, max_tokens=120):
    resp = anthropic_client.messages.create(
        model=ANTHROPIC_HAIKU,
        max_tokens=max_tokens,
        system=LORA_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )
    return resp.content[0].text.strip()


def lora_call(user_message, max_tokens=120):
    r = requests.post(endpoint_url, json={"user_message": user_message, "max_tokens": max_tokens}, timeout=120)
    r.raise_for_status()
    return r.json()["text"]


def primary_pick():
    # YOUR CODE: counter-based split. Increment REQUEST_COUNTER["n"] by 1.
    # If lora_ratio is 0.0, return "haiku". Otherwise compute step = round(1 / ratio).
    # If REQUEST_COUNTER["n"] % step == 0, return "lora"; else "haiku".
    # This guarantees the long-run ratio matches exactly.
    raise NotImplementedError("Replace with the counter-based pick.")


SAMPLE_QUERIES = [
    "What is Widget?",
    "How do I cancel my Widget subscription?",
    "Does Widget integrate with Slack?",
    "How much is the team plan?",
    "Is there an API rate limit?",
] * 40  # 200 total


def run_load_test(prompts):
    rows = []
    for q in prompts:
        request_id = str(uuid.uuid4())
        primary_model = primary_pick()
        try:
            if primary_model == "lora":
                primary_response = lora_call(q)
            else:
                primary_response = haiku_call(q)
        except Exception as exc:
            primary_response = f"[ERROR: {exc!r}]"
        rows.append({
            "request_id": request_id,
            "user_message": q,
            "primary_model": primary_model,
            "primary_response": primary_response,
            "shadow_model": None,
            "shadow_response": None,
            "shadow_model_responded": False,
            "judge_verdict": None,
        })
    # Batched insert.
    for i in range(0, len(rows), 100):
        supabase.table(TRAFFIC_LOG_TABLE).insert(rows[i:i+100]).execute()
    return rows


load_rows = run_load_test(SAMPLE_QUERIES)
lora_count = sum(1 for r in load_rows if r["primary_model"] == "lora")
print(f"Load test: {len(load_rows)} requests. LoRA share: {lora_count}/{len(load_rows)} = {lora_count/len(load_rows):.1%}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: LoRA share in [9%, 11%] over the 200-request load test.


def checkpoint_2(session):
    rows = supabase.table(TRAFFIC_LOG_TABLE).select("primary_model").execute().data
    if len(rows) < 200:
        return CheckpointResult(status="fail", error_reason=f"Expected 200 traffic_log rows; got {len(rows)}.")
    lora = sum(1 for r in rows if r["primary_model"] == "lora")
    share = lora / len(rows)
    if not (0.09 <= share <= 0.11):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"LoRA share = {share:.1%}; expected between 9% and 11%. "
                f"Random-based splits drift; the lab requires counter-based (every Nth request)."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Random splits land within +/- 4pp on 200 requests; the test requires +/- 1pp. Use a counter.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def primary_pick():
    REQUEST_COUNTER["n"] += 1
    ratio = SPLIT_CONFIG["lora_ratio"]
    if ratio <= 0:
        return "haiku"
    step = max(1, round(1 / ratio))
    return "lora" if REQUEST_COUNTER["n"] % step == 0 else "haiku"
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** 200 Haiku-or-LoRA primary calls. ~180 Haiku at $0.80/$4 per 1M, ~20 LoRA on Modal. ~$0.50 cumulative: ~$0.65.

## Task 3: Shadow traffic on every request

For every request, the OTHER model also gets called. The primary's response is returned; the shadow's response is logged. Use `asyncio.gather` to parallelize so the shadow does not double the user-facing latency.

Update the traffic_log rows with `shadow_model`, `shadow_response`, `shadow_model_responded=true`. The validator confirms all 200 rows have `shadow_model_responded=true`.

In [ ]:
# Task 3: shadow traffic.

import asyncio


async def call_haiku_async(user_message):
    return await asyncio.to_thread(haiku_call, user_message)


async def call_lora_async(user_message):
    return await asyncio.to_thread(lora_call, user_message)


async def fan_out_shadow(user_message, primary_model):
    """Returns (primary_response, shadow_model, shadow_response)."""
    if primary_model == "lora":
        # primary is LoRA; shadow is Haiku.
        primary_task = call_lora_async(user_message)
        shadow_task = call_haiku_async(user_message)
        shadow_model = "haiku"
    else:
        primary_task = call_haiku_async(user_message)
        shadow_task = call_lora_async(user_message)
        shadow_model = "lora"
    # YOUR CODE: await asyncio.gather(primary_task, shadow_task, return_exceptions=True).
    # Return (primary_response, shadow_model, shadow_response). On exception in
    # either, return the repr() as the response so the row still inserts.
    raise NotImplementedError("Replace with the asyncio.gather + unpack.")


# Update every traffic_log row by running the shadow for the matching request.
async def shadow_pass(rows):
    sem = asyncio.Semaphore(8)  # cap parallelism so Modal does not OOM

    async def one(row):
        async with sem:
            primary_resp, shadow_model, shadow_resp = await fan_out_shadow(
                row["user_message"], row["primary_model"]
            )
            return {
                "request_id": row["request_id"],
                "primary_response": primary_resp,
                "shadow_model": shadow_model,
                "shadow_response": shadow_resp,
                "shadow_model_responded": True,
            }

    tasks = [one(r) for r in rows]
    return await asyncio.gather(*tasks)


updates = asyncio.run(shadow_pass(load_rows))
# Patch each row by request_id; supabase-py does not have a bulk update-by-key,
# so we do it one by one.
for u in updates:
    supabase.table(TRAFFIC_LOG_TABLE).update(
        {
            "primary_response": u["primary_response"],
            "shadow_model": u["shadow_model"],
            "shadow_response": u["shadow_response"],
            "shadow_model_responded": True,
        }
    ).eq("request_id", u["request_id"]).execute()
print(f"Shadow pass done. {len(updates)} rows updated.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: all 200 rows have shadow_model_responded=True.


def checkpoint_3(session):
    rows = supabase.table(TRAFFIC_LOG_TABLE).select("shadow_model_responded,shadow_model").execute().data
    not_responded = sum(1 for r in rows if not r["shadow_model_responded"])
    if not_responded > 0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{not_responded} rows still have shadow_model_responded=False. "
                f"The shadow pass either skipped them or asyncio.gather raised on a subset."
            ),
        )
    # Sanity: shadow_model must be the opposite of primary on every row.
    primary_models = {r.get("primary_model") for r in supabase.table(TRAFFIC_LOG_TABLE).select("primary_model").execute().data}
    if not primary_models.issubset({"lora", "haiku"}):
        return CheckpointResult(status="fail", error_reason=f"Unexpected primary_model values: {primary_models}.")
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

`asyncio.gather(*tasks, return_exceptions=True)` returns a list of results in the same order as the input tasks. Index `[0]` is the primary, `[1]` is the shadow.

</details>

<details><summary>Hint 2 (direction)</summary>

```
async def fan_out_shadow(user_message, primary_model):
    if primary_model == "lora":
        primary_task = call_lora_async(user_message)
        shadow_task = call_haiku_async(user_message)
        shadow_model = "haiku"
    else:
        primary_task = call_haiku_async(user_message)
        shadow_task = call_lora_async(user_message)
        shadow_model = "lora"
    results = await asyncio.gather(primary_task, shadow_task, return_exceptions=True)
    primary_resp = results[0] if not isinstance(results[0], Exception) else repr(results[0])
    shadow_resp = results[1] if not isinstance(results[1], Exception) else repr(results[1])
    return primary_resp, shadow_model, shadow_resp
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Shadow pass: 200 more LLM calls (the one we did NOT do as primary on each request). Roughly 20 LoRA calls on Modal plus 180 more Haiku. ~$0.40. Cumulative: ~$1.

## Task 4: LLM-as-judge classifies agreement

For every traffic_log row, call Claude Sonnet with both responses + the user message and a rubric that returns `agree` or `disagree`. Write the verdict to `judge_verdict`. For `disagree` rows, also insert a row into the disagreements table.

The validator confirms every row has `judge_verdict` populated and the disagreements table has at least one row (unless the corpus is small enough that Haiku and the LoRA happened to fully agree).

In [ ]:
# Task 4: LLM judge.

JUDGE_RUBRIC = (
    "You are evaluating whether two AI responses to the same user message convey the same answer. "
    "Reply with the single word 'agree' if the responses convey the same factual answer (style "
    "differences are acceptable), otherwise reply with 'disagree'."
)


def judge_pair(user_message, primary_response, shadow_response):
    # YOUR CODE: call Sonnet with system=JUDGE_RUBRIC, max_tokens=8, and a user
    # message containing the three strings. Parse the response: if it contains
    # "disagree" return "disagree", else "agree".
    raise NotImplementedError("Replace with the Sonnet call + parser.")


# Pull all rows; judge each; update.
all_rows = supabase.table(TRAFFIC_LOG_TABLE).select(
    "request_id,user_message,primary_response,shadow_response"
).execute().data

judge_updates = []
disagreement_inserts = []
for r in all_rows:
    verdict = judge_pair(r["user_message"], r["primary_response"], r["shadow_response"])
    judge_updates.append({"request_id": r["request_id"], "judge_verdict": verdict})
    if verdict == "disagree":
        disagreement_inserts.append({
            "request_id": r["request_id"],
            "user_message": r["user_message"],
            "primary_response": r["primary_response"],
            "shadow_response": r["shadow_response"],
            "judge_rationale": "Verdict from Sonnet judge.",
        })

for u in judge_updates:
    supabase.table(TRAFFIC_LOG_TABLE).update({"judge_verdict": u["judge_verdict"]}).eq("request_id", u["request_id"]).execute()
if disagreement_inserts:
    for i in range(0, len(disagreement_inserts), 50):
        supabase.table(DISAGREEMENTS_TABLE).insert(disagreement_inserts[i:i+50]).execute()

print(f"Judged {len(judge_updates)} rows. Disagreements: {len(disagreement_inserts)}.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: every traffic_log row has judge_verdict; disagreements table reflects the verdicts.


def checkpoint_4(session):
    rows = supabase.table(TRAFFIC_LOG_TABLE).select("judge_verdict").execute().data
    if any(r["judge_verdict"] is None for r in rows):
        missing = sum(1 for r in rows if r["judge_verdict"] is None)
        return CheckpointResult(
            status="fail",
            error_reason=f"{missing} traffic_log rows still have judge_verdict=NULL.",
        )
    actual_disagree = sum(1 for r in rows if r["judge_verdict"] == "disagree")
    table_disagree = supabase.table(DISAGREEMENTS_TABLE).select("id", count="exact").limit(1).execute().count
    if actual_disagree > 0 and table_disagree == 0:
        return CheckpointResult(
            status="fail",
            error_reason=f"{actual_disagree} rows judged disagree but the disagreements table is empty.",
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The judge call is the standard Sonnet messages API. Parse for the literal "disagree" substring; default to "agree".

</details>

<details><summary>Hint 2 (direction)</summary>

```
def judge_pair(user_message, primary_response, shadow_response):
    resp = anthropic_client.messages.create(
        model=ANTHROPIC_SONNET, max_tokens=8, system=JUDGE_RUBRIC,
        messages=[{"role": "user", "content": (
            f"USER: {user_message}\nPRIMARY: {primary_response}\nSHADOW: {shadow_response}\nVerdict:"
        )}],
    )
    text = resp.content[0].text.strip().lower()
    return "disagree" if "disagree" in text else "agree"
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** 200 Sonnet judge calls at ~100 input tokens each: roughly $0.50. Cumulative: ~$1.50.

## Task 5: Flip the traffic split to 0% in one config change

**The killer moment.** You mutate `SPLIT_CONFIG["lora_ratio"] = 0.0` and re-run 50 requests through the router. None of them should go to LoRA. The validator queries the traffic_log for rows inserted AFTER the flip and confirms `primary_model="lora"` count is 0.

In [ ]:
# Task 5: flip.

print(f"Before flip: SPLIT_CONFIG = {SPLIT_CONFIG}")

# YOUR CODE: mutate SPLIT_CONFIG["lora_ratio"] to 0.0.
raise NotImplementedError("Replace with the one-line flip.")

print(f"After flip: SPLIT_CONFIG = {SPLIT_CONFIG}")

# Run 50 more requests; tag them with a phase marker so the validator can isolate.
FLIP_TAG = "post_flip"
post_flip_rows = []
for q in SAMPLE_QUERIES[:50]:
    request_id = f"flip-{uuid.uuid4()}"
    primary_model = primary_pick()
    try:
        if primary_model == "lora":
            primary_response = lora_call(q)
        else:
            primary_response = haiku_call(q)
    except Exception as exc:
        primary_response = f"[ERROR: {exc!r}]"
    post_flip_rows.append({
        "request_id": request_id,
        "user_message": q,
        "primary_model": primary_model,
        "primary_response": primary_response,
        "shadow_model": FLIP_TAG,
        "shadow_response": "n/a",
        "shadow_model_responded": True,
        "judge_verdict": "n/a",
    })
supabase.table(TRAFFIC_LOG_TABLE).insert(post_flip_rows).execute()
post_flip_lora = sum(1 for r in post_flip_rows if r["primary_model"] == "lora")
print(f"Post-flip LoRA count: {post_flip_lora}/50")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: post-flip rows have primary_model="lora" count == 0.


def checkpoint_5(session):
    rows = supabase.table(TRAFFIC_LOG_TABLE).select("primary_model").eq("shadow_model", "post_flip").execute().data
    lora_after = sum(1 for r in rows if r["primary_model"] == "lora")
    if lora_after != 0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{lora_after} post-flip rows still routed to LoRA. The router must read SPLIT_CONFIG "
                f"at decision time, not at module-import time. Confirm primary_pick reads from "
                f"SPLIT_CONFIG['lora_ratio'] on every call."
            ),
        )
    if len(rows) < 50:
        return CheckpointResult(status="fail", error_reason=f"Expected 50 post-flip rows; found {len(rows)}.")
    return CheckpointResult(status="pass")


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

One line: assign 0.0 to the dict. The router will pick it up because `primary_pick` reads the dict on every call.

</details>

<details><summary>Hint 2 (direction)</summary>

```
SPLIT_CONFIG["lora_ratio"] = 0.0
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2. If CP5 fails, `primary_pick` likely captured the ratio at module-import time; rewrite to read from the dict on every invocation.

</details>

**Wallet check.** 50 Haiku calls only (no LoRA). ~$0.05. Cumulative: ~$1.55.

## Cleanup

Stop the Modal app (replicas drop immediately), delete the app and volume, clear Supabase tables, drop the local summary.

In [ ]:
# NBVAL_SKIP
# Persist summary BEFORE cleanup so the cleanup card sees it.
all_traffic = supabase.table(TRAFFIC_LOG_TABLE).select("primary_model,judge_verdict").execute().data
summary = {
    "total_requests": len(all_traffic),
    "lora_share": sum(1 for r in all_traffic if r["primary_model"] == "lora") / max(1, len(all_traffic)),
    "disagreements": sum(1 for r in all_traffic if r["judge_verdict"] == "disagree"),
}
with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=1)

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type == "modal_app")
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(1 for fid in failed_ids if MODAL_APP_NAME in fid)
standard_destroyed = standard_total - (len(failed_ids) - (critical_total - critical_destroyed))
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, Modal or Supabase state remains. Resolve before closing.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $3 to $4.50.** Modal L4 dominated (cold-load + warm-pool + load test); Haiku was the cheap baseline; Sonnet judge was the second-largest line. Modal app, Volume, Supabase rows, and local summary all cleared.

## Reflection

These are not graded. They are for you.

1. The traffic split held within 1pp over 200 requests. At 200K requests per day, what is the expected drift, and what monitoring would catch a misconfiguration that broke the split?

2. The LLM judge flagged some disagreements out of 200. The product owner asks: "Are these disagreements acceptable to ship?" Write the two sentences you would send back, naming what data you would gather first.

## Exam-Style Practice

**Question 1 (A/B split discipline):**

A new fine-tuned model is shipping to 10% traffic with shadow eval. The shadow eval reports 8% disagreement rate. The product owner asks if this is safe to roll out. What is the right answer?

A. Safe to roll out; 92% agreement is high.
B. Need to inspect the 8%; agreement rate alone does not tell you which model is right when they disagree.
C. Roll back; 8% is too high.
D. Increase traffic to 50% to gather more data.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: agreement is not equivalent to correctness; both models can agree on the wrong answer or disagree where the new model is right.
- B is correct: the action item is to sample disagreements, score them on quality, and decide based on the directional signal (is the new model right more often than the old?).
- C is premature without inspection.
- D scales the risk before understanding it.

</details>

**Question 2 (rollback primitive):**

A new model is at 10% traffic. A spike in complaints lands. What is the right rollback primitive?

A. Stop the Modal app.
B. Flip the traffic split to 0%.
C. Redeploy the previous model.
D. Restart the routing service.

<details><summary>Show answer</summary>

**Correct: B.**

- A stops the app but breaks the 10% of traffic that was still in flight.
- B is correct: flipping the ratio to 0% is the canonical rollback. The old model keeps serving; the new model stops getting traffic; the Modal app idles down on its own.
- C re-creates infra; slower.
- D does not fix the routing; it just restarts it.

</details>

**Question 3 (shadow latency):**

A shadow pass doubles the LLM cost per request. The team asks if shadow should run on every request. What is the right answer?

A. Yes, always; shadow eval is the only way to detect regressions.
B. No, sample at 1-5%; shadow on every request is wasteful for stable models.
C. Yes, but only for paid users.
D. Yes, but with a smaller shadow model.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wasteful at scale; 100% shadow on a 100K-QPS service is a separate budget line.
- B is correct: 1-5% sampling is the production norm; raise the sample rate when you suspect drift.
- C ties experiment design to billing; weird incentive.
- D weakens the signal; the shadow's purpose is to be an honest comparator.

</details>